# DiFaReli++ - TargetSH - Total Score

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

data = pd.read_csv('./targetSH_MJ/Batch_5346588_batch_results.csv')
df = data[['Input.sj_name', 'Input.vs_name', 'WorkTimeInSeconds', 'WorkerId', 'Answer.taskAnswers']]

out_q1 = {
    'difareli++': 0,
    'hou22_geom': 0,
    'hou21_shadowm': 0,
    'ic_light': 0,
    'diffusionrig': 0,
    'relipa': 0,
    'total_relighting': 0,
}
out_q2 = {
    'difareli++': 0,
    'hou22_geom': 0,
    'hou21_shadowm': 0,
    'ic_light': 0,
    'diffusionrig': 0,
    'relipa': 0,
    'total_relighting': 0,
}
for i in range(len(df)):
    answer = json.loads(df.iloc[i]['Answer.taskAnswers'])
    assert len(answer) == 1
    answer = answer[0]
    # print(list(answer.keys())[:20])
    # print(list(answer.keys())[20:])
    for a in list(answer.keys())[:20]:
        tmp = [x for x, y in answer[a].items() if y is True]
        assert len(tmp) == 1
        qidx = int(a.split('_')[0][1:])
        user_ans = tmp[0][3:]
        assert qidx != 0
        if qidx % 2 != 0:
            out_q1[user_ans] += 1
        else: out_q2[user_ans] += 1
        
n_out_q1 = sum([v for v in out_q1.values()])
n_out_q2 = sum([v for v in out_q2.values()])

total = {}
for k in out_q1.keys():
    total[k] = out_q1[k] + out_q2[k]
    
print("Total q1 answer: ", n_out_q1)
print("Total q2 answer: ", n_out_q2)
print("="*100)
print("Answer: ")
print("Q1: ", out_q1)
print("Q2: ", out_q2)
print("Total: ", total)

perc_out_q1 = {k: (v/n_out_q1)*100 for k, v in out_q1.items()}
perc_out_q2 = {k: (v/n_out_q2)*100 for k, v in out_q2.items()}
perc_out_total = {k: (v/(n_out_q1+n_out_q2))*100 for k, v in total.items()}

print("="*100)
print("Percentage: ")
print("Q1: ", perc_out_q1)
print("Q2: ", perc_out_q2)
print("Total: ", perc_out_total)


Total q1 answer:  360
Total q2 answer:  360
Answer: 
Q1:  {'difareli++': 180, 'hou22_geom': 28, 'hou21_shadowm': 22, 'ic_light': 40, 'diffusionrig': 44, 'relipa': 25, 'total_relighting': 21}
Q2:  {'difareli++': 187, 'hou22_geom': 32, 'hou21_shadowm': 24, 'ic_light': 42, 'diffusionrig': 28, 'relipa': 23, 'total_relighting': 24}
Total:  {'difareli++': 367, 'hou22_geom': 60, 'hou21_shadowm': 46, 'ic_light': 82, 'diffusionrig': 72, 'relipa': 48, 'total_relighting': 45}
Percentage: 
Q1:  {'difareli++': 50.0, 'hou22_geom': 7.777777777777778, 'hou21_shadowm': 6.111111111111111, 'ic_light': 11.11111111111111, 'diffusionrig': 12.222222222222221, 'relipa': 6.944444444444445, 'total_relighting': 5.833333333333333}
Q2:  {'difareli++': 51.94444444444445, 'hou22_geom': 8.88888888888889, 'hou21_shadowm': 6.666666666666667, 'ic_light': 11.666666666666666, 'diffusionrig': 7.777777777777778, 'relipa': 6.388888888888888, 'total_relighting': 6.666666666666667}
Total:  {'difareli++': 50.97222222222222, 'ho

# DiFaReli++ - TargetSH - 1Vs1 Score

In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

def proc(fn, out_q1_vs, out_q2_vs):
    data = pd.read_csv(fn)
    df = data[['Input.sj_name', 'Input.vs_name', 'WorkTimeInSeconds', 'WorkerId', 'Answer.taskAnswers']]

    # Mapping q{idx}_p{id} to be the same as out_q{1,2}_vs.keys()

    for i in range(len(df)):
        answer = json.loads(df.iloc[i]['Answer.taskAnswers'])
        sj_name = df.iloc[i]['Input.sj_name'].split('#')
        vs_name = df.iloc[i]['Input.vs_name'].split('#')
        
        mapping = {}
        count = 0
        for idx, sj in enumerate(sj_name):
            keyname = f"q{count+1}_p{sj.replace('pair', '')}"
            mapping[keyname] = vs_name[idx]
            keyname = f"q{count+2}_p{sj.replace('pair', '')}"
            mapping[keyname] = vs_name[idx]
            count += 2
        
        assert len(answer) == 1
        answer = answer[0]
        # print(answer)
        
        for a in list(answer.keys())[:20]:
            assert len([x for x, y in answer[a].items() if y is True]) == 1
            tmp = [x for x, y in answer[a].items()]
            # print(a, tmp)
            vs_mapping = mapping[a]
            # print(vs_mapping)
            user_ans_1 = tmp[0][3:]
            user_ans_2 = tmp[1][3:]
            # print(user_ans_0, user_ans_1)
            # print(answer[a][f'a1_{user_ans_0}'], answer[a][f'a2_{user_ans_1}'])
            # print(out_q1_vs[vs_mapping])
            qidx = int(a.split('_')[0][1:])
            assert qidx != 0
            if qidx % 2 != 0:
                # print(out_q1_vs[vs_mapping], vs_mapping, answer[a], user_ans_1, user_ans_2)
                out_q1_vs[vs_mapping][user_ans_1] += int(answer[a][f'a1_{user_ans_1}'])
                out_q1_vs[vs_mapping][user_ans_2] += int(answer[a][f'a2_{user_ans_2}'])
                # print(out_q1_vs[vs_mapping], vs_mapping, answer[a], user_ans_1, user_ans_2)
                # print("="*50)
            else:
                out_q2_vs[vs_mapping][user_ans_1] += int(answer[a][f'a1_{user_ans_1}'])
                out_q2_vs[vs_mapping][user_ans_2] += int(answer[a][f'a2_{user_ans_2}'])
        
    return out_q1_vs, out_q2_vs
    assert False
            
    n_out_q1 = sum([v for v in out_q1.values()])
    n_out_q2 = sum([v for v in out_q2.values()])

    total = {}
    for k in out_q1.keys():
        total[k] = out_q1[k] + out_q2[k]
        
    print("Total q1 answer: ", n_out_q1)
    print("Total q2 answer: ", n_out_q2)
    print("="*100)
    print("Answer: ")
    print("Q1: ", out_q1)
    print("Q2: ", out_q2)
    print("Total: ", total)

    perc_out_q1 = {k: (v/n_out_q1)*100 for k, v in out_q1.items()}
    perc_out_q2 = {k: (v/n_out_q2)*100 for k, v in out_q2.items()}
    perc_out_total = {k: (v/(n_out_q1+n_out_q2))*100 for k, v in total.items()}

    print("="*100)
    print("Percentage: ")
    print("Q1: ", perc_out_q1)
    print("Q2: ", perc_out_q2)
    print("Total: ", perc_out_total)
    
vs_list = ['difareli++-vs-ic_light', 'difareli++-vs-hou21_shadowm', 'difareli++-vs-relipa', 'difareli++-vs-diffusionrig', 'difareli++-vs-hou22_geom']
out_q1_vs = {}
out_q2_vs = {}
for v in vs_list:
    ours = 'difareli++'
    splits = v.split('-vs-')
    assert ours == splits[0]
    theirs = splits[1]
    out_q1_vs[v] = {ours: 0, theirs: 0}
    out_q2_vs[v] = {ours: 0, theirs: 0}

out_q1_vs, out_q2_vs = proc('./rotateSH_MJ/Batch_5346586_batch_results.csv', out_q1_vs, out_q2_vs)
out_q1_vs, out_q2_vs = proc('./rotateSH_MJ/Batch_5346605_batch_results.csv', out_q1_vs, out_q2_vs)

# Sumarize results for each 1vs1 comparison
for k in out_q1_vs.keys():
    print(f"[#] Pair: {k}")
    ours = k.split('-vs-')[0]
    theirs = k.split('-vs-')[1]
    ours_score_q1 = out_q1_vs[k][ours]
    theirs_score_q1 = out_q1_vs[k][theirs]
    total_q1 = ours_score_q1 + theirs_score_q1
    ours_perc_q1 = (ours_score_q1/total_q1)*100
    theirs_perc_q1 = (theirs_score_q1/total_q1)*100
    print(f"\tQ1 - Ours vs Theirs: {ours_score_q1} vs {theirs_score_q1}")
    print(f"\tQ1 - Ours vs Theirs (%): {ours_perc_q1:.2f}% vs {theirs_perc_q1:.2f}%")
    
    ours_score_q2 = out_q2_vs[k][ours]
    theirs_score_q2 = out_q2_vs[k][theirs]
    total_q2 = ours_score_q2 + theirs_score_q2
    ours_perc_q2 = (ours_score_q2/total_q2)*100
    theirs_perc_q2 = (theirs_score_q2/total_q2)*100
    print(f"\tQ2 - Ours vs Theirs: {ours_score_q2} vs {theirs_score_q2}")
    print(f"\tQ2 - Ours vs Theirs (%): {ours_perc_q2:.2f}% vs {theirs_perc_q2:.2f}%")
    print("="*100)

[#] Pair: difareli++-vs-ic_light
	Q1 - Ours vs Theirs: 198 vs 102
	Q1 - Ours vs Theirs (%): 66.00% vs 34.00%
	Q2 - Ours vs Theirs: 215 vs 85
	Q2 - Ours vs Theirs (%): 71.67% vs 28.33%
[#] Pair: difareli++-vs-hou21_shadowm
	Q1 - Ours vs Theirs: 220 vs 80
	Q1 - Ours vs Theirs (%): 73.33% vs 26.67%
	Q2 - Ours vs Theirs: 204 vs 96
	Q2 - Ours vs Theirs (%): 68.00% vs 32.00%
[#] Pair: difareli++-vs-relipa
	Q1 - Ours vs Theirs: 228 vs 72
	Q1 - Ours vs Theirs (%): 76.00% vs 24.00%
	Q2 - Ours vs Theirs: 238 vs 62
	Q2 - Ours vs Theirs (%): 79.33% vs 20.67%
[#] Pair: difareli++-vs-diffusionrig
	Q1 - Ours vs Theirs: 219 vs 81
	Q1 - Ours vs Theirs (%): 73.00% vs 27.00%
	Q2 - Ours vs Theirs: 215 vs 85
	Q2 - Ours vs Theirs (%): 71.67% vs 28.33%
[#] Pair: difareli++-vs-hou22_geom
	Q1 - Ours vs Theirs: 208 vs 92
	Q1 - Ours vs Theirs (%): 69.33% vs 30.67%
	Q2 - Ours vs Theirs: 205 vs 95
	Q2 - Ours vs Theirs (%): 68.33% vs 31.67%
